Building a GPT
Companion notebook to the Zero To Hero video on GPT.

In [24]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
print(f"Length of dataset in characters: {len(text)}")

--2026-02-05 13:48:29--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2026-02-05 13:48:30 (15.4 MB/s) - ‘input.txt.1’ saved [1115394/1115394]

Length of dataset in characters: 1115394


Now we have to build a vocab i.e., the list of all the  chars we have in our data set

In [25]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("All the unique characters:", ''.join(chars))

All the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz



Simple Encoder and decoder for tokenising

So during enumeration we need to assign the index to a vocab i.e., the set of chars and special chars we have in our dataset

In [34]:
x = { ch:i for i,ch in enumerate(chars) }
y = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [ x[ch] for ch in s ]
decode = lambda l: ''.join( y[i] for i in l )

# print( encode('vinayak') )
# print( decode(encode('vinayak')) )

There are multiple tokenisation algorithms , this is one simple case. Google uses [sentiencepiece](https://github.com/google/sentencepiece)
OpenAI uses [openAItiktoken](https://github.com/openai/tiktoken) this is what GPT uses

So in practice, sometimes having a very long sequence of tokens is resource intensive , here for example we are using char level encoding hence we get very long encoded seq of numbers so its best to use sub word encoding like the above lib use.






In [27]:
# Let's now encode the entire dataset and store it in a torch tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10])



torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


Lets split our data into train and validation test

In [28]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

Now we cannot feed the entire text into our transformer that would be computationally expensive, instead we sample random chucks for training , these are sometimes called blocks or blocksize or context length or window. Let's define this

In [29]:
block_size = 8 # context length: how many characters do we look at when predicting the next one?
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [30]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context.tolist()} the target: {target.item()}")

When input is [18] the target: 47
When input is [18, 47] the target: 56
When input is [18, 47, 56] the target: 57
When input is [18, 47, 56, 57] the target: 58
When input is [18, 47, 56, 57, 58] the target: 1
When input is [18, 47, 56, 57, 58, 1] the target: 15
When input is [18, 47, 56, 57, 58, 1, 15] the target: 47
When input is [18, 47, 56, 57, 58, 1, 15, 47] the target: 58


Now that we have defined the time dimension that is the block size which is basically used by transformer to perform the prediction as we in the above code, we have to define batch dimension since GPUs are good at parallel processing so all batches does the same block size prediction independently.

Note: The torch.manual_seed() function in the PyTorch library sets the seed for the random number generator (RNG) across all devices (CPU and CUDA GPUs) to ensure reproducibility of results

Here we are sampling random locations in our datasets to pull chunks from we set manual_seed to ensure reproducibility when i run this anywhere.

torch.randint generates random offsets of length batch_size between 0 and len of data - block_size

torch.stack takes one d arrays and stacks them as rows in a 4 by 8 tensor.

In [31]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print("----")


for b in range(batch_size):  # batch dimension
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context.tolist()} the target: {target.item()}")

tensor([ 76049, 234249, 934904, 560986])
inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
When input is [24] the target: 43
When input is [24, 43] the target: 58
When input is [24, 43, 58] the target: 5
When input is [24, 43, 58, 5] the target: 57
When input is [24, 43, 58, 5, 57] the target: 1
When input is [24, 43, 58, 5, 57, 1] the target: 46
When input is [24, 43, 58, 5, 57, 1, 46] the target: 43
When input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
When input is [44] the target: 53
When input is [44, 53] the target: 56
When input is [44, 53, 56] the target: 1
When input is [44, 53, 56, 1] the target: 58
When input is [44, 53, 56, 1,

In [32]:
print(xb) #our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


Lets start with a simple transformer in terms of a language model it is a bigram language model. So here the idx lets say is 24 will go to the embedding table and pluck out the 24th row. logits are a score for the next char in the seq. Basically we are predicting what comes next based on the individual identity of a single token.

Now lets measure the loss function since we have predicted the logits , a good way to measure a loss or a quality of a measurement is use the negative log likelyhood loss which is implemented in pytorch as cross_entropy.

In [35]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch,Time,Channel)
        
        # cross entropy loss expects the input to be (Batch,Channel,Time)
        if targets is None:
            loss = None
        else:
    
            logits = logits.view(-1, logits.size(-1))
            targets = targets.view(-1)
            
            loss = F.cross_entropy(logits, targets)
        
        return logits,loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B,C)
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) # append to the running sequence
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # -ln(1/65)


print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [37]:
# pytorch Optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [48]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(f"loss {loss.item()}")

tensor([279420, 200535, 360890, 697338, 349221, 598217, 734482, 232237, 673164,
         53049, 961450, 531611, 308952, 565513, 604092,  93574, 485184, 988613,
        469550, 503527, 854979, 308373,  50071, 635925, 336593,  94870, 116929,
        697655, 224640, 944336, 719259, 922770])
tensor([ 926986,  506463,  855745,   47417,  952146,  315608,  679937,  515719,
         143057,  803009,  117058,  620660,  389279,  690998,  975806,  120447,
         628698,  903576,  559837,  143087,  755088,  619679,  752819,  467591,
         714855, 1002026,  330506,  970176,  916167,   10727,  418832,  515615])
tensor([ 86646, 256026, 985193, 781339, 753820, 991386, 340744, 901064, 384806,
         56500, 509314,  47514,   4087,  76246, 623429, 915496, 605908, 467717,
        997789, 742531, 458407, 303720, 483055, 682188, 199930,  41005, 968163,
        549152, 748007, 284866, 832398, 678611])
tensor([ 42196, 346467, 242340, 297734, 749066, 184130, 146799, 270166, 525853,
        133585, 33421

In [50]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


Maldimyomy,
ENGRiveanthes woanefell ICa helfasth t He chat:
KI lll y. theeieagim ho t ts m
He thooupo yo bot co-ave surit.
Bu'Bee TINAst Langrin t swe s ff atormesaldslondsay ndeve ndnear. foula ancade ted ELondiserdoundd?
T:
Mitho yonteam s? ouiearsithe ghormm l beritold KI:

e IChieeng nslalllly w
